In [2]:
import pandas as pd
import numpy as np

# ==========================================
# 1. Data Loading
# ==========================================
col_names = ['unit_nr', 'time_cycles', 'setting_1', 'setting_2', 'setting_3']
col_names += [f's_{i}' for i in range(1, 22)]

# header=None indicates columns are indexed as 0, 1, 2, etc., before mapping col_names
train_df = pd.read_csv('../data/train_FD001.txt', sep=r'\s+', header=None, names=col_names)

# ==========================================
# 2. Target Engineering (Remaining Useful Life - RUL)
# ==========================================
max_cycle = train_df.groupby('unit_nr')['time_cycles'].max().reset_index()
max_cycle.columns = ['unit_nr', 'max_cycle']
train_df = train_df.merge(max_cycle, on=['unit_nr'], how='left')

# RUL is the difference between the maximum cycle and the current cycle
train_df['RUL'] = train_df['max_cycle'] - train_df['time_cycles']
train_df.drop('max_cycle', axis=1, inplace=True)

# ==========================================
# 3. Noise Removal (Dropping Constant Sensors)
# ==========================================
# Transpose (.T) to switch rows to columns for easier filtering
stats = train_df.describe().T

# Collect columns where standard deviation is exactly zero
constant_columns = stats[stats['std'] == 0].index.tolist()
train_df.drop(columns=constant_columns, inplace=True)

# ==========================================
# 4. Temporal Feature Engineering
# ==========================================
sensor_cols = [col for col in train_df.columns if col.startswith('s_')]
window = 15

for col in sensor_cols:
    # Rolling Mean
    train_df[f'{col}_mean_{window}'] = train_df.groupby('unit_nr')[col].transform(
        lambda x: x.rolling(window, min_periods=1).mean()
    )
    # Rolling Standard Deviation
    train_df[f'{col}_std_{window}'] = train_df.groupby('unit_nr')[col].transform(
        lambda x: x.rolling(window, min_periods=1).std().fillna(0)
    )

In [3]:
from sklearn.preprocessing import StandardScaler

# ==========================================
# 5. Data Standardization (Scaling)
# ==========================================
# Identify columns to scale (exclude unit_nr, time_cycles, and RUL)
columns_to_scale = [col for col in train_df.columns if col not in ['unit_nr', 'time_cycles', 'RUL']]

scaler = StandardScaler()
# Apply fit_transform only on the selected feature columns
train_df[columns_to_scale] = scaler.fit_transform(train_df[columns_to_scale])

# ==========================================
# 6. Final Evaluation & Export
# ==========================================
print(f"Final data shape ready for training: {train_df.shape}")

processed_file_path = '../data/train_FD001_processed.csv'
train_df.to_csv(processed_file_path, index=False)
print(f"[SUCCESS] Pipeline executed. Processed data saved to: {processed_file_path}")

Final data shape ready for training: (20631, 56)
[SUCCESS] Pipeline executed. Processed data saved to: ../data/train_FD001_processed.csv


In [ ]:
# IMPORTANT: We must save the scaler used during training
# Without the exact same scaler, the production API will fail
import joblib
joblib.dump(scaler, 'scaler.joblib')

['scaler.joblib']